In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Ruta de tu dataset con las características ya extraídas
ruta_dataset = '/content/drive/MyDrive/Proyecto/Daily_Sports_Activities/Data/dataset_final_features.csv'

# Cargar el dataset
df = pd.read_csv(ruta_dataset)

print(f"Dimensiones del dataset: {df.shape}")
df.head()

Mounted at /content/drive
Dimensiones del dataset: (15200, 243)


,activity,subject,window_id,T_acc_mean_x,T_acc_mean_y,T_acc_mean_z,T_acc_std_x,T_acc_std_y,T_acc_std_z,T_acc_max_x,...,LL_mag_max_x,LL_mag_max_y,LL_mag_max_z,LL_mag_corr_xy,LL_mag_corr_xz,LL_mag_corr_yz,LL_mag_mag_mean,LL_mag_mag_std,LL_mag_mag_auc,LL_mag_mag_mean_diff
0,1,1,0,8.015509,1.058076,5.553903,0.129444,0.039797,0.191729,8.1605,...,0.74182,0.30267,-0.055365,-0.380922,0.214412,-0.094971,0.800508,0.000745,2.369513,0.000941
1,1,1,1,7.920071,1.126935,5.683707,0.058170,0.026639,0.105984,8.0412,...,0.74320,0.30342,-0.054963,-0.351583,0.448888,-0.306916,0.801040,0.000701,2.371086,0.000761
2,1,1,2,8.001183,1.141395,5.559029,0.095242,0.030720,0.148445,8.1763,...,0.74335,0.30377,-0.054945,-0.231525,0.377849,-0.342104,0.801930,0.000862,2.373707,0.000829
3,1,1,3,7.941989,1.143843,5.658659,0.059360,0.024328,0.094272,8.1160,...,0.74302,0.30397,-0.054711,-0.266598,0.365792,-0.255355,0.802269,0.000731,2.374725,0.000797
4,1,1,4,7.996011,1.138048,5.567233,0.042821,0.021047,0.067826,8.0860,...,0.74316,0.30423,-0.055413,-0.169517,0.621999,-0.270246,0.802356,0.000820,2.375000,0.000938


In [3]:
print("--- Hold-Out Independiente del Sujeto ---")

# En Daily and Sports Activities los sujetos van del 1 al 8.
# Asignamos uno a prueba y otro a validación.
sujetos_para_test = [8]
sujetos_para_validacion = [7]

# Máscaras
mask_test = df['subject'].isin(sujetos_para_test)
mask_val = df['subject'].isin(sujetos_para_validacion)

# Train = todo excepto test y validación
mask_train = ~(mask_test | mask_val)

# Conjuntos (Eliminamos 'activity', 'subject' y 'window_id')
X_train_A = df[mask_train].drop(columns=['activity', 'subject', 'window_id'])
y_train_A = df[mask_train]['activity']

X_val_A = df[mask_val].drop(columns=['activity', 'subject', 'window_id'])
y_val_A = df[mask_val]['activity']

X_test_A = df[mask_test].drop(columns=['activity', 'subject', 'window_id'])
y_test_A = df[mask_test]['activity']

# Información
print(f"Sujetos en Train      : {df[mask_train]['subject'].unique()}")
print(f"Sujetos en Validación : {df[mask_val]['subject'].unique()}")
print(f"Sujetos en Test       : {df[mask_test]['subject'].unique()}")

print(f"Forma de X_train_A: {X_train_A.shape}")
print(f"Forma de X_val_A  : {X_val_A.shape}")
print(f"Forma de X_test_A : {X_test_A.shape}")

--- Hold-Out Independiente del Sujeto ---
Sujetos en Train      : [1 2 3 4 5 6]
Sujetos en Validación : [7]
Sujetos en Test       : [8]
Forma de X_train_A: (11400, 240)
Forma de X_val_A  : (1900, 240)
Forma de X_test_A : (1900, 240)


In [4]:
nombres_19_actividades = [
    'Sitting', 'Standing', 'Lying on back', 'Lying on right side',
    'Ascending stairs', 'Descending stairs', 'Standing still in elevator',
    'Moving around in elevator', 'Walking in parking lot', 'Walking on treadmill (flat)',
    'Walking on treadmill (15 deg)', 'Running on treadmill', 'Exercising on stepper',
    'Exercising on cross trainer', 'Cycling on exercise bike (horizontal)',
    'Cycling on exercise bike (vertical)', 'Rowing', 'Jumping', 'Playing basketball'
]

In [5]:
from sklearn.preprocessing import StandardScaler

print("--- Estandarización de los datos (StandardScaler) ---")

scaler = StandardScaler()

# 1. El scaler SE AJUSTA (fit) ÚNICAMENTE con los datos de entrenamiento
# para aprender la media y la desviación estándar sin hacer trampa.
# Inmediatamente después, transforma esos mismos datos.
X_train_A = scaler.fit_transform(X_train_A)

# 2. Transforma los datos de validación y prueba usando las reglas
# matemáticas que aprendió solo del conjunto de entrenamiento.
X_val_A = scaler.transform(X_val_A)
X_test_A = scaler.transform(X_test_A)

print(f"Estandarización completada.")
print(f"Media de X_train_A (aprox 0): {X_train_A.mean():.5f}")
print(f"Varianza de X_train_A (aprox 1): {X_train_A.var():.5f}")

--- Estandarización de los datos (StandardScaler) ---
Estandarización completada.
Media de X_train_A (aprox 0): 0.00000
Varianza de X_train_A (aprox 1): 1.00000


In [6]:
from sklearn.decomposition import KernelPCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
import pandas as pd
from itertools import product

In [7]:
classifiers = {
    "Logistic": LogisticRegression(max_iter=200, C=1, penalty='l2', solver='lbfgs')
}

In [8]:
# KPCA RBF
gamma_list = [1e-3, 1e-2, 1e-1]
n_components_list = [32, 64, 128]

# KPCA Polinomial
degree_list = [1, 2, 3, 4]

In [9]:
def evaluar_modelo(X_train, X_test, y_train, y_test, modelo):
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    return accuracy_score(y_test, y_pred)

###Bucle para Kernel RBF

In [10]:
resultados_kpca = []

for gamma, n_comp in product(gamma_list, n_components_list):

    print(f"RBF | gamma={gamma} | n_components={n_comp}")

    # KPCA
    kpca = KernelPCA(
        n_components=n_comp,
        kernel='rbf',
        gamma=gamma,
        fit_inverse_transform=False,
        n_jobs=-1,
        random_state=42
    )

    # Fit SOLO en train
    X_train_k = kpca.fit_transform(X_train_A)
    X_val_k  = kpca.transform(X_val_A)

    # Evaluar clasificadores
    for nombre_clf, clf in classifiers.items():
        acc = evaluar_modelo(X_train_k, X_val_k, y_train_A, y_val_A, clf)

        resultados_kpca.append({
            "kernel": "rbf",
            "gamma": gamma,
            "degree": None,
            "n_components": n_comp,
            "classifier": nombre_clf,
            "accuracy": acc
        })

RBF | gamma=0.001 | n_components=32
RBF | gamma=0.001 | n_components=64
RBF | gamma=0.001 | n_components=128
RBF | gamma=0.01 | n_components=32
RBF | gamma=0.01 | n_components=64
RBF | gamma=0.01 | n_components=128
RBF | gamma=0.1 | n_components=32
RBF | gamma=0.1 | n_components=64
RBF | gamma=0.1 | n_components=128


###Bucle para Kernel Polinomial:

In [11]:
for degree, n_comp in product(degree_list, n_components_list):

    print(f"Poly | degree={degree} | n_components={n_comp}")

    kpca = KernelPCA(
        n_components=n_comp,
        kernel='poly',
        degree=degree,
        coef0=0,
        gamma=1,
        n_jobs=-1,
        random_state=42
    )

    X_train_k = kpca.fit_transform(X_train_A)
    X_val_k  = kpca.transform(X_val_A)

    for nombre_clf, clf in classifiers.items():
        acc = evaluar_modelo(X_train_k, X_val_k, y_train_A, y_val_A, clf)

        resultados_kpca.append({
            "kernel": "poly",
            "gamma": None,
            "degree": degree,
            "n_components": n_comp,
            "classifier": nombre_clf,
            "accuracy": acc
        })

Poly | degree=1 | n_components=32


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Poly | degree=1 | n_components=64
Poly | degree=1 | n_components=128
Poly | degree=2 | n_components=32


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Poly | degree=2 | n_components=64


...

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Poly | degree=4 | n_components=64


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Poly | degree=4 | n_components=128


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [12]:
df_resultados_kpca = pd.DataFrame(resultados_kpca)

df_resultados_kpca.sort_values(by="accuracy", ascending=False).head(20)

,kernel,gamma,degree,n_components,classifier,accuracy
9,poly,NaN,1.0,32,Logistic,0.956842
0,rbf,0.001,NaN,32,Logistic,0.955263
13,poly,NaN,2.0,64,Logistic,0.946316
2,rbf,0.001,NaN,128,Logistic,0.935263
10,poly,NaN,1.0,64,Logistic,0.931579
1,rbf,0.001,NaN,64,Logistic,0.928947
11,poly,NaN,1.0,128,Logistic,0.924737
17,poly,NaN,3.0,128,Logistic,0.921579
16,poly,NaN,3.0,64,Logistic,0.913684
14,poly,NaN,2.0,128,Logistic,0.890000


In [13]:
df_rbf = df_resultados_kpca[df_resultados_kpca["kernel"] == "rbf"]

df_rbf.groupby(
    ["gamma", "n_components"]
)["accuracy"].mean().sort_values(ascending=False).head(10)

gamma  n_components
0.001  32              0.955263
       128             0.935263
       64              0.928947
0.010  64              0.741053
       32              0.729474
       128             0.723684
0.100  32              0.054211
       64              0.052632
       128             0.052632
Name: accuracy, dtype: float64

In [14]:
for clf in df_resultados_kpca["classifier"].unique():
    print(f"\nClasificador: {clf}")

    df_clf = df_resultados_kpca[df_resultados_kpca["classifier"] == clf]

    display(
        df_clf.sort_values(by="accuracy", ascending=False)
        .style.highlight_max(subset=["accuracy"], color="lightgreen")
    )


Clasificador: Logistic


,kernel,gamma,degree,n_components,classifier,accuracy
9,poly,nan,1.000000,32,Logistic,0.956842
0,rbf,0.001000,nan,32,Logistic,0.955263
13,poly,nan,2.000000,64,Logistic,0.946316
2,rbf,0.001000,nan,128,Logistic,0.935263
10,poly,nan,1.000000,64,Logistic,0.931579
1,rbf,0.001000,nan,64,Logistic,0.928947
11,poly,nan,1.000000,128,Logistic,0.924737
17,poly,nan,3.000000,128,Logistic,0.921579
16,poly,nan,3.000000,64,Logistic,0.913684
14,poly,nan,2.000000,128,Logistic,0.890000
